
##### DEPRECATED
##### Replaced by: Bronze_SQL

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import input_file_name
import sys, os
from utils.defs import get_file

spark = SparkSession.builder \
    .appName("Load Data Bronze") \
    .config("spark.sql.shuffle.partitions", "200")  \
    .config("spark.sql.files.maxPartitionBytes", "128MB") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

In [0]:
path = "/Volumes/vendas_pecas/default/my/"
data = "processar/"
bronze = "bronze/"
bronze_schema = "vendas_pecas.camada_bronze"

#METATDADOS DO ARQUIVO

In [0]:
arquivo_mais_recente = get_file(path+data)

In [0]:
df_vendas = spark.read.option("header", "true").csv(path + data + arquivo_mais_recente).select("*", col("_metadata.file_path").alias("file_path"), col("_metadata.file_modification_time").alias("ingest_date"))

In [0]:
df_vendas.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(bronze_schema + ".tb_bronze")
#APPEND - bronze sempre incremental